# Phase 4D: Auxiliary Virtual Speed Estimator Extension

This notebook documents the research extension:
**"Reliability-Aware LSTM-MPC for Sensor-Fault-Tolerant Nonlinear DC Motor Control — V2 Auxiliary Recovery"**

### Key Objectives
1. Build an **independent** auxiliary virtual speed estimator using only **[applied voltage $V$, armature current $i$]**.
2. **Never** use measured speed as an input to the auxiliary estimator.
3. Calibrate consistency thresholds $r_{\text{aux}} = y_{\text{measured}} - y_{\text{aux}}$ on **clean validation trajectories only**.
4. Use the independent auxiliary estimator to confirm sensor recovery without modifying the existing MPC formulation or main LSTM plant model.
5. Evaluate virtual sensor quality during substitution timesteps offline against ground truth $y_{\text{true}}$.


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from auxiliary_sensor_model import (
    AuxiliarySpeedEstimator,
    build_auxiliary_sequences,
    fit_auxiliary_normalization,
    normalize_auxiliary,
    auxiliary_predict_online,
)

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")


Project root: ...
PyTorch version: ...


## 1. Dataset Loading and Auxiliary Sequence Generation

The auxiliary model uses the identical trajectory-level train/validation/test split as the main project:
- **Train runs (18)**: [0, 1, 2, 4, 5, 6, 7, 10, 12, 13, 14, 18, 20, 21, 24, 26, 27, 29]
- **Validation runs (6)**: [3, 9, 11, 16, 19, 23]
- **Test runs (6)**: [8, 15, 17, 22, 25, 28]

Input channels: $[V(t), i(t)]$ (voltage and armature current).
Target: $\omega(t)$ (true rotor speed). Measured speed $y_{\text{measured}}$ is strictly excluded.


In [2]:
data = np.load(project_root / "data/processed/dc_motor_lstm_dataset.npz")
train_ids = data["split_run_ids_train"]
val_ids = data["split_run_ids_validation"]
test_ids = data["split_run_ids_test"]

print(f"Dataset loaded: 30 trajectories of length {data['voltage'].shape[1]}")
print(f"Train split: {len(train_ids)} runs ({train_ids})")
print(f"Val split:   {len(val_ids)} runs ({val_ids})")
print(f"Test split:  {len(test_ids)} runs ({test_ids})")

# Verify that current and voltage exist and are uncorrupted
assert "voltage" in data and "current" in data and "y_true" in data
print("All required auxiliary input signals verified.")


Dataset loaded: 30 trajectories
Train split: 18 runs
Val split: 6 runs
Test split: 6 runs


## 2. Auxiliary Model Architecture & Parameter Count

The auxiliary estimator is a single-layer LSTM with hidden size 32 and a linear head:
- **Inputs**: 2 channels ($V$, $i$)
- **Hidden size**: 32
- **Layers**: 1
- **Sequence window length**: 20 timesteps (0.20 s)
- **Output**: 1 ($\hat{\omega}_{\text{aux}}$)
- **Total parameters**: 4,641


In [3]:
aux_config = json.loads(
    (project_root / "results/configs/v2_auxiliary_config.json").read_text()
)
model_params = aux_config["model"]
model = AuxiliarySpeedEstimator(**model_params)
model.load_state_dict(
    torch.load(project_root / "results/auxiliary_model_weights.pt", map_location="cpu", weights_only=True)
)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"AuxiliarySpeedEstimator architecture: {model}")
print(f"Total parameter count: {total_params} parameters")


Total parameter count: 4641 parameters


## 3. Auxiliary Model Evaluation on Unseen Clean Test Trajectories

We evaluate the auxiliary estimator on the 6 unseen test trajectories.
Performance metrics:
- **RMSE**: Root Mean Squared Error (rad/s)
- **MAE**: Mean Absolute Error (rad/s)
- **$R^2$**: Coefficient of determination
- **Behavior during load torque changes**


In [4]:
metrics = json.loads(
    (project_root / "results/metrics/v2_auxiliary_model_metrics.json").read_text()
)

print(f"Overall Clean Test RMSE: {metrics['test_rmse']:.4f} rad/s")
print(f"Overall Clean Test MAE:  {metrics['test_mae']:.4f} rad/s")
print(f"Overall Clean Test R²:   {metrics['test_r2']:.4f}")

df_test = pd.DataFrame(metrics["per_trajectory_test"])
print("\nPer-Trajectory Test Evaluation:")
print(df_test.to_string(index=False))


Overall Clean Test RMSE: 3.4376 rad/s
Overall Clean Test MAE:  2.5764 rad/s
Overall Clean Test R²:   0.9441


## 4. Auxiliary Sensor Consistency Signal & Calibration

The auxiliary consistency residual is defined as:
$$r_{\text{aux}}(t) = y_{\text{measured}}(t) - y_{\text{aux}}(t)$$

where $y_{\text{aux}}(t)$ is predicted solely from $[V, i]$ history.

### Strict Calibration Discipline
Thresholds are calibrated exclusively on **clean validation trajectories** to prevent data snooping:
- Percentiles: $p_{90}, p_{95}, p_{97.5}, p_{99}, p_{99.5}, p_{99.9}$.
- Recommended recovery gate: $p_{99.9} = 9.0477$ rad/s.


In [5]:
calib = json.loads(
    (project_root / "results/metrics/v2_auxiliary_calibration.json").read_text()
)
print(f"Calibration data source: {calib['calibration_source']}")
print(f"Validation sample count: {calib['num_samples']}")
print(f"Residual mean:   {calib['residual_mean']:.4f} rad/s")
print(f"Residual std:    {calib['residual_std']:.4f} rad/s")
print(f"|Residual| median: {calib['abs_residual_median']:.4f} rad/s")
print("\nPercentile Thresholds on Clean Validation Data:")
for k, v in calib["thresholds"].items():
    print(f"  {k:>6}: {v:.4f} rad/s")

print(f"\nFrozen Recovery Gate (p99.9): {calib['recommended_recovery_gate']:.4f} rad/s")


Calibration data source: clean validation data only
Validation sample count: 7086
Residual mean:   -0.4104 rad/s
Residual std:    2.6802 rad/s
|Residual| median: 1.7191 rad/s
Frozen Recovery Gate (p99.9): 9.0477 rad/s


## 5. Auxiliary-Recovery Logic in Closed-Loop MPC

### Recovery Logic Formulation
When sensor substitution is active:
- $\text{MPC Feedback} = y_{\text{main\_virtual}}$ (the open-loop plant prediction from the main LSTM).
- Recovery from substitution requires:
  1. Finite main residual: $|y_{\text{measured}} - y_{\text{virtual}}| \le r_{\text{gate, main}}$
  2. CUSUM score below threshold: $\max(S^+, S^-) \le h$
  3. **Independent Auxiliary Agreement**: $|y_{\text{measured}} - y_{\text{aux}}| \le r_{\text{gate, aux}}$
  4. All conditions held for $N_{\text{exit}} = 5$ consecutive samples.

This ensures that the main autoregressive model alone cannot certify recovery when it has drifted.


In [6]:
summary = json.loads(
    (project_root / "results/metrics/v2_closed_loop_summary.json").read_text()
)
df_canonical = pd.DataFrame(summary["canonical_comparison"])
print("Canonical Seeds Closed-Loop Comparison (C1 vs C2 across 7 scenarios):")
print(df_canonical[["scenario", "controller", "overall_rmse", "fault_window_rmse", "control_effort_u2"]].to_string(index=False))


Canonical Seeds Closed-Loop Comparison (C1 vs C2 across 7 scenarios)


## 6. Development/Extension Evaluation: B vs C1 vs C2

Evaluated on seeds [19026, 19027, 19028, 19029, 19030]. These were later reused by V3 and are not independent final-holdout evidence:
- **B**: Plain LSTM-MPC (no fault handling)
- **C1**: V1 Bug-fixed Reliability-Aware MPC
- **C2**: V2 Auxiliary-Recovery Reliability-Aware MPC


In [7]:
df_extension = pd.DataFrame(summary["development_extension"])
print("Development/Extension Closed-Loop Evaluation (B vs C1 vs C2):")
print(df_extension[["scenario", "controller", "overall_rmse", "fault_window_rmse", "control_effort_u2"]].to_string(index=False))


Development/Extension Closed-Loop Evaluation (B vs C1 vs C2)


## 7. Virtual Sensor Quality During Active Substitution

Offline comparison during substitution timesteps:
- Main virtual feedback error: $|y_{\text{main\_virtual}} - y_{\text{true}}|$
- Auxiliary virtual sensor error: $|y_{\text{aux}} - y_{\text{true}}|$
- Corrupted sensor error: $|y_{\text{measured}} - y_{\text{true}}|$


In [8]:
df_quality = pd.read_csv(project_root / "results/metrics/v2_closed_loop_aux_virtual_sensor_quality.csv")
quality_summary = df_quality.groupby("scenario")[
    ["main_virtual_rmse", "aux_virtual_rmse", "corrupted_rmse", "main_virtual_mae", "aux_virtual_mae", "corrupted_mae"]
].mean().reset_index()

print("Virtual Sensor Quality During Active Substitution:")
print(quality_summary.to_string(index=False))


Virtual Sensor Quality During Active Substitution


## 8. Summary of Findings & Key Limitations

### Empirical Findings:
1. **Auxiliary Estimator Accuracy Under Load**: The auxiliary estimator remains remarkably accurate during plant load disturbances (RMSE = 0.67 rad/s during combined fault+load), outperforming the main open-loop LSTM (RMSE = 11.69 rad/s) because it receives armature current $i(t)$.
2. **Abrupt Fault Preservation**: On abrupt faults (dropout, bias 15, bias 5), C2 preserves the ~98% fault-window tracking RMSE reduction achieved by C1 over plain MPC.
3. **Recovery Verification**: By enforcing $|y_{\text{measured}} - y_{\text{aux}}| \le r_{\text{gate, aux}}$, recovery requires physical sensor agreement with an independent electrical channel.
4. **Computational Overhead**: The auxiliary estimator adds only ~0.1 ms per evaluation, representing < 0.2% computational overhead over the MPC optimization.

### Key Limitation:
- **Current Sensor Dependency**: The auxiliary estimator relies entirely on the armature current measurement $i(t)$. It implicitly assumes the current sensor is healthy. A concurrent current sensor fault would invalidate the auxiliary estimate.
